In [ ]:
import pandas as pd
import json
import collections
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from tqdm import tqdm
import anndata as ad
import numpy as np
import seaborn as sns
from pathlib import Path

In [ ]:
METADATA_DIR = "../../../data/immunotherapy_cohorts/IMMotion150/EGAD00001004183-metadata"

# run_sample.csv has semicolon-separated key=value attributes — easier to parse than samples.csv JSON
run_sample = pd.read_csv(f"{METADATA_DIR}/run_sample.csv")

def parse_sample_attributes(attr_str):
    attrs = {}
    if pd.isna(attr_str):
        return attrs
    for item in str(attr_str).split(";"):
        if "=" in item:
            k, v = item.split("=", 1)
            attrs[k.strip()] = v.strip()
    return attrs

attrs_df = run_sample["sample_attributes"].apply(parse_sample_attributes)
attrs_df = pd.DataFrame(list(attrs_df))

# Combine with run/sample accession IDs
meta = pd.concat([run_sample[["sample_alias", "sample_accession_id", "run_accession_id"]], attrs_df], axis=1)

# RNA-seq samples are those with AnnoRNASampleID present
rna_meta = meta[meta["AnnoRNASampleID"].notna()].copy()

# Cast numeric fields
for col in ["PFS", "CNSR_PFS", "TMB", "TNB", "CD8A_IHC", "CD31_IHC", "ICSCORE"]:
    rna_meta[col] = pd.to_numeric(rna_meta[col], errors="coerce")

# Canonical response label: responder = CR or PR
rna_meta["response_binary"] = rna_meta["BestResponse"].map(
    {"CR": 1, "PR": 1, "SD": 0, "PD": 0}  # NE → NaN automatically
)

print(f"Total samples in metadata: {len(meta)}")
print(f"RNA-seq samples:           {len(rna_meta)}")
print(f"  with PFS + CNSR_PFS:     {rna_meta['PFS'].notna().sum()}")
print(f"  with BestResponse:       {rna_meta['BestResponse'].notna().sum()}")
print(f"  with binary label (excl. NE): {rna_meta['response_binary'].notna().sum()}")

rna_meta.head(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# --- BestResponse by ARM ---
ax = axes[0]
response_order = ["CR", "PR", "SD", "PD", "NE"]
arm_order = ["Atezo+Bev", "Atezo", "Sunitinib"]
resp_counts = (
    rna_meta[rna_meta["BestResponse"].notna()]
    .groupby(["ARM", "BestResponse"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=arm_order, columns=response_order, fill_value=0)
)
colors = {"CR": "#2ca02c", "PR": "#98df8a", "SD": "#ffbb78", "PD": "#d62728", "NE": "#aec7e8"}
resp_counts.plot(kind="bar", stacked=True, ax=ax,
                 color=[colors[c] for c in resp_counts.columns], legend=True, rot=0)
ax.set_title("Best Response by ARM")
ax.set_xlabel("ARM")
ax.set_ylabel("Number of RNA-seq samples")
ax.legend(title="Response", bbox_to_anchor=(1.01, 1), loc="upper left")
for c in ax.containers:
    ax.bar_label(c, label_type="center", fmt="%d", fontsize=8,
                 labels=[v if v > 0 else "" for v in c.datavalues])

# --- ORR (CR+PR) by ARM ---
ax = axes[1]
orr = rna_meta.groupby("ARM").apply(
    lambda x: pd.Series({
        "ORR (CR+PR)": (x["BestResponse"].isin(["CR", "PR"])).sum() / x["BestResponse"].notna().sum() * 100,
        "n": x["BestResponse"].notna().sum()
    })
).reindex(arm_order)
bars = ax.bar(orr.index, orr["ORR (CR+PR)"], color=["#1f77b4", "#ff7f0e", "#2ca02c"])
ax.set_title("Objective Response Rate (ORR) by ARM")
ax.set_ylabel("ORR (%)")
ax.set_ylim(0, 60)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))
for bar, (arm, row) in zip(bars, orr.iterrows()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f"{row['ORR (CR+PR)']:.1f}%\n(n={int(row['n'])})", ha="center", va="bottom", fontsize=9)

# --- PFS distribution by ARM ---
ax = axes[2]
for arm, color in zip(arm_order, ["#1f77b4", "#ff7f0e", "#2ca02c"]):
    subset = rna_meta[(rna_meta["ARM"] == arm) & rna_meta["PFS"].notna()]["PFS"]
    ax.hist(subset, bins=20, alpha=0.6, label=f"{arm} (n={len(subset)})", color=color)
ax.set_title("PFS Distribution by ARM")
ax.set_xlabel("PFS (months)")
ax.set_ylabel("Count")
ax.legend()

plt.tight_layout()
plt.savefig("IMMotion150_cohort_summary.png", dpi=150, bbox_inches="tight")
plt.show()

# Tabular summary
summary = rna_meta.groupby("ARM").agg(
    n=("AnnoRNASampleID", "count"),
    CR=("BestResponse", lambda x: (x == "CR").sum()),
    PR=("BestResponse", lambda x: (x == "PR").sum()),
    SD=("BestResponse", lambda x: (x == "SD").sum()),
    PD=("BestResponse", lambda x: (x == "PD").sum()),
    NE=("BestResponse", lambda x: (x == "NE").sum()),
    missing_response=("BestResponse", lambda x: x.isna().sum()),
    has_PFS=("PFS", lambda x: x.notna().sum()),
    median_PFS=("PFS", "median"),
    pfs_event_rate=("CNSR_PFS", lambda x: (x == 0).sum() / x.notna().sum()),
).reindex(arm_order)
summary["ORR (%)"] = (summary["CR"] + summary["PR"]) / (summary["n"] - summary["NE"] - summary["missing_response"]) * 100
print(summary.to_string())

In [ ]:
# Join RNA-seq sample accession IDs with file accession IDs (EGAFs)
sample_file = pd.read_csv(f"{METADATA_DIR}/sample_file.csv")

rna_accessions = set(rna_meta["sample_accession_id"])
rna_files = sample_file[sample_file["sample_accession_id"].isin(rna_accessions)].copy()

In [ ]:
rna_files.head()

In [ ]:
# Get entries in file_name where sample_accession_id is EGAN00001749284
rna_file = rna_files[rna_files["sample_accession_id"] == "EGAN00001749284"]

In [ ]:
rna_file['file_name'].values[0]

In [ ]:
# print the full file_name field, rn it's getting cut off in the display
rna_file['file_name'].values[1]

In [ ]:
# Join RNA-seq sample accession IDs with file accession IDs (EGAFs)
sample_file = pd.read_csv(f"{METADATA_DIR}/sample_file.csv")

rna_accessions = set(rna_meta["sample_accession_id"])
rna_files = sample_file[sample_file["sample_accession_id"].isin(rna_accessions)].copy()

print(f"RNA-seq samples:  {len(rna_accessions)}")
print(f"Fastq files (2 per sample, paired-end): {len(rna_files)}")
print(f"Unique EGAF IDs:  {rna_files['file_accession_id'].nunique()}")
print()

# Save EGAF IDs for download
# Command
# cat /data/rbg/users/ashyamal/RNA-FM-benchmarking/data/immunotherapy_cohorts/IMMotion150/RNA/rna_seq_egaf_ids.txt | \
#xargs -P 8 -I {} pyega3 -cf /data/rbg/users/ashyamal/credentials.json -c 2 fetch \
#    --output-dir /data/rbg/users/ashyamal/RNA-FM-benchmarking/data/immunotherapy_cohorts/IMMotion150/RNA \
#    {}
OUT_DIR = "../../../data/immunotherapy_cohorts/IMMotion150/RNA"
egaf_path = f"{OUT_DIR}/rna_seq_egaf_ids.txt"
rna_files["file_accession_id"].to_csv(egaf_path, index=False, header=False)
print(f"Saved {len(rna_files)} EGAF IDs to {egaf_path}")

# Also save a full manifest with sample alias, ARM, BestResponse for easy cross-referencing
manifest = rna_files.merge(
    rna_meta[["sample_accession_id", "AnnoRNASampleID", "AnonPatID", "ARM",
               "BestResponse", "response_binary", "PFS", "CNSR_PFS", "MSKCC", "Stage"]],
    on="sample_accession_id", how="left"
).sort_values(["AnnoRNASampleID", "file_name"])

manifest_path = f"{OUT_DIR}/rna_seq_download_manifest.csv"
manifest.to_csv(manifest_path, index=False)
print(f"Saved download manifest to {manifest_path}")
manifest.head(6)

In [ ]:
import subprocess

egaf_path = f"{OUT_DIR}/rna_seq_egaf_ids.txt"
dataset_files_path = f"{METADATA_DIR}/dataset_files.txt"

result = subprocess.run(
    f"grep -Ff {egaf_path} {dataset_files_path} | awk '{{sum += $6}} END {{printf \"%.2f\", sum / 1e9}}'",
    shell=True, capture_output=True, text=True
)
print(f"Total RNA-seq download size: {result.stdout} GB")


In [ ]:
# Number of samples with both RNA-seq and responder information that's note NE
labeled = rna_meta[rna_meta["response_binary"].notna()]
print(f"Samples with RNA-seq + responder/non-responder label (excl. NE): {len(labeled)}")
print(f"  Responders   (CR/PR): {(labeled['response_binary'] == 1).sum()}")
print(f"  Non-responders (SD/PD): {(labeled['response_binary'] == 0).sum()}")

In [ ]:
import os
import shutil
from pathlib import Path

RNA_DIR = Path("../../../data/immunotherapy_cohorts/IMMotion150/RNA")
manifest = pd.read_csv(RNA_DIR / "rna_seq_download_manifest.csv")

skipped, moved, already_done = [], [], []

for sample_id, grp in manifest.groupby("AnnoRNASampleID"):
    sample_dir = RNA_DIR / "paired" /sample_id
    sample_dir.mkdir(exist_ok=True)

    for _, row in grp.iterrows():
        egaf_dir = RNA_DIR / row["file_accession_id"]

        # pyega3 strips the long EGA prefix — glob for the actual file
        fastq_files = [f for f in egaf_dir.glob("*.fastq.gz") if not f.name.endswith(".md5")]
        if not fastq_files:
            skipped.append(str(egaf_dir))
            continue

        src = fastq_files[0]
        read_num = "R1" if src.name.endswith("_1.fastq.gz") else "R2"
        dst = sample_dir / f"{sample_id}_{read_num}.fastq.gz"

        if dst.exists():
            already_done.append(str(dst))
            continue

        shutil.move(str(src), str(dst))
        moved.append(str(dst))

print(f"Moved:        {len(moved)} files")
print(f"Already done: {len(already_done)} files")
print(f"Not found:    {len(skipped)} EGAF dirs (not downloaded yet)")
if skipped:
    print("  First few missing:", skipped[:5])

### Analyze Processed RNA

In [ ]:
RNA_DIR = Path("../../../data/immunotherapy_cohorts/IMMotion150/RNA")
adata = ad.read_h5ad(RNA_DIR / "IMMotion150_bulk_RNA_processed_log1p.h5ad")
print(adata)

In [ ]:
# ── Basic counts ──────────────────────────────────────────────────────────────
print(f"Samples: {adata.n_obs}   Genes: {adata.n_vars}")
print(f"\nResponse label:\n{adata.obs['response_label'].value_counts()}")
print(f"\nARM:\n{adata.obs['ARM'].value_counts()}")
print(f"\nMSKCC risk:\n{adata.obs['MSKCC'].value_counts()}")
print(f"\nStage:\n{adata.obs['Stage'].value_counts()}")
print(f"\nMissing response: {adata.obs['response_label'].isna().sum()}")

In [ ]:
# ── Response by ARM ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

adata.obs["response_label"].value_counts().plot.bar(ax=axes[0], color=["#DD8452","#4C72B0"], rot=0)
axes[0].set_title("Response (R/NR)")
axes[0].set_ylabel("Count")

ct = pd.crosstab(adata.obs["ARM"], adata.obs["response_label"])
ct.plot.bar(stacked=True, ax=axes[1], color=["#DD8452","#4C72B0"], rot=15)
axes[1].set_title("Response by ARM")
axes[1].set_ylabel("Count")

adata.obs["PFS"].hist(bins=30, ax=axes[2], color="#4C72B0", edgecolor="none")
axes[2].set_title("PFS distribution")
axes[2].set_xlabel("PFS (months)")

plt.suptitle("IMMotion150 cohort (KIRC)", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Expression QC ─────────────────────────────────────────────────────────────
X = adata.X

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Per-gene log-variance
axes[0].hist(adata.var["log_variance"], bins=80, color="#4C72B0", edgecolor="none")
axes[0].set_title("Per-gene log-variance")
axes[0].set_xlabel("log-variance")

# Per-sample total expression
sample_totals = X.sum(axis=1)
axes[1].hist(sample_totals, bins=40, color="#55A868", edgecolor="none")
axes[1].set_title("Per-sample total expression (log1p)")
axes[1].set_xlabel("Sum of log1p TPM")

# Per-sample fraction of zero genes
zero_frac = (X == 0).mean(axis=1)
axes[2].hist(zero_frac, bins=40, color="#DD8452", edgecolor="none")
axes[2].set_title("Per-sample zero-gene fraction")
axes[2].set_xlabel("Fraction of genes = 0")

plt.suptitle("IMMotion150: expression QC", fontsize=12)
plt.tight_layout()
plt.show()

print(f"Mean per-sample total expression: {sample_totals.mean():.1f}")
print(f"Mean zero-gene fraction:          {zero_frac.mean():.3f}")
print(f"Genes with log_variance > 1:      {(adata.var['log_variance'] > 1).sum()}")

In [ ]:
# ── Overlap with other ICI cohorts ────────────────────────────────────────────
other_cohorts = {}
for name in ["Hugo", "Riaz", "Rose"]:
    p = Path(f"../../../data/immunotherapy_cohorts/{name}/RNA/{name}_bulk_RNA_processed_log1p.h5ad")
    if p.exists():
        other_cohorts[name] = set(ad.read_h5ad(p).var_names)

imm_genes = set(adata.var_names)
print(f"IMMotion150 genes: {len(imm_genes)}")
for name, genes in other_cohorts.items():
    shared = len(imm_genes & genes)
    print(f"  Shared with {name}: {shared} / {min(len(imm_genes), len(genes))}")

In [ ]:
adata_raw = ad.read_h5ad(RNA_DIR / "IMMotion150_bulk_RNA_processed.h5ad")
raw_tpm_sums = adata_raw.X.sum(axis=1)
print(f"Raw TPM sum per sample — mean: {raw_tpm_sums.mean():.0f}  min: {raw_tpm_sums.min():.0f}  max: {raw_tpm_sums.max():.0f}")
# Expect ~600k-900k (less than 1M because we filtered to protein-coding genes only)

In [ ]:
low_thresh = 400000  # flag samples below this
low_samples = adata_raw.obs_names[raw_tpm_sums < low_thresh]
print(f"Samples with raw TPM sum < 400k: {len(low_samples)}")
for sid in low_samples:
    resp = adata_raw.obs.loc[sid, 'response_label']
    tpm_sum = raw_tpm_sums[adata_raw.obs_names.get_loc(sid)]
    print(f"  {sid}  TPM sum={tpm_sum:.0f}  response={resp}")
